In [6]:
import numpy as np
import pandas as pd
import concurrent.futures
import timeit
from functools import partial
from copy import deepcopy
import sys
from IPython.display import display
# Add measurement mcts python package to path
sys.path.append('../src/measurement_mcts')
from measurement_mcts.mcts.mcts import mcts_with_rollout
from measurement_mcts.mcts.tree_viz import render_pyvis
from measurement_mcts.state_evaluation.hertg import HERTG
from measurement_mcts.environment.measurement_control_env import MeasurementControlEnvironment

# Create the environment
env = MeasurementControlEnvironment(init_reset=False)

Toy Measurement Control Initialized


In [2]:
def get_percent_done(state, env):
    # First calculate total trace at start
    total_trace = env.init_covariance_trace
    final_coner_trace = env.final_corner_cov_trace
    
    # Then calculate trace of ooi cov
    ooi_covs = state[2]
    per_ooi_traces = np.trace(ooi_covs, axis1=2, axis2=3)
    all_traces = per_ooi_traces.flatten()
    
    # Make any traces below the final corner trace 0
    all_traces[all_traces <= final_coner_trace] = 0
    print(all_traces)
    
    # Sum the traces and calculate the percentage
    sum_traces = np.sum(all_traces)
    print(f'sum_traces: {sum_traces}')
    return (1 - sum_traces / total_trace) * 100

In [3]:
def get_mcts_metrics(state, object_true_state, max_actions=200, LI=100,
                     EF=0.1, DF=1.0, rollout_method='random_same',
                     hertg_method='static', rollout_pre_collision_stop=True) -> dict:
    """
    Run MCTS and return the metrics.
    params:
        state: the initial state of the environment
        object_true_state: the true state of the object
        max_actions: the maximum number of actions to take
        LI: the length of the interval for the HERTG method
        EF: the exploration factor for the HERTG method
        DF: the discount factor for the HERTG method
        rollout_method: the rollout method to use
        hertg_method: the HERTG method to use
        
    returns:
        metrics: a dictionary of metrics
    """
    
    # Create environment and HERTG objects
    env = MeasurementControlEnvironment(init_reset=False)
    env.object_manager.set_true_state(object_true_state)
    hertg = HERTG(state, env, method=hertg_method)
    
    # Create metric trackers
    cumulative_reward = 0.
    num_actions = 0
    done = False
    start_time = timeit.default_timer()
    for i in range(max_actions):
        # Run MCTS and take the best action
        root = mcts_with_rollout(env, state, LI, EF, DF, rollout_method,
                                 rollout_pre_collision_stop, hertg=hertg)
        best_action_idx = np.argmax(root.child_Q())
        state, reward, done = env.step(state, env.action_space[best_action_idx])
        
        # Reset the horizon to 0
        state_list = list(state)
        state_list[3] = 0
        state = tuple(state_list)
        
        # Increment the cumulative reward and number of actions
        cumulative_reward += reward
        num_actions = i + 1
        if done:
            break
        
    comp_time = timeit.default_timer() - start_time
    percent_done = get_percent_done(state, env)
    
    metrics = {
        'LI': LI,
        'EF': EF,
        'DF': DF,
        'rollout_method': rollout_method,
        'hertg_method': hertg_method,
        'done': done,
        'percent_done': percent_done,
        'cumulative_reward': cumulative_reward,
        'num_actions': num_actions,
        'computation_time': comp_time,
        'computation_per_action': comp_time / num_actions,
    }
    
    return metrics

In [4]:
state = env.reset()
object_true_state = env.object_manager.get_true_state()
get_mcts_metrics(state, object_true_state)

Toy Measurement Control Initialized


c:\Users\austi\Documents\CodeScratch\MeasurementMCTS\metrics\../src/measurement_mcts\measurement_mcts\mcts\mcts.py:222: RuntimeWarning: divide by zero encountered in log
  * np.sqrt(np.log(self.number_visits)


[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0


{'LI': 100,
 'EF': 0.1,
 'DF': 1.0,
 'rollout_method': 'random_same',
 'hertg_method': 'static',
 'done': True,
 'percent_done': 100.0,
 'cumulative_reward': 6.920210396026119,
 'num_actions': 84,
 'computation_time': 53.443608800007496,
 'computation_per_action': 0.6362334380953273}

In [8]:
# -----------------------------------------------------------------------------
# Worker wrapper: creates a fresh environment and runs one trial of the metrics.
# -----------------------------------------------------------------------------
def worker_wrapper(rollout_method,
                   max_actions=200,
                   LI=100,
                   EF=0.1,
                   DF=1.0,
                   hertg_method='static',
                   rollout_pre_collision_stop=True):
    """
    Creates a fresh environment, resets it to obtain a new random state, and
    runs the MCTS metrics collection with the given rollout method.
    """
    # Create a fresh environment instance.
    # (Assuming that MeasurementControlEnvironment() properly initializes the environment.)
    # env = MeasurementControlEnvironment()
    
    # Get a random starting state and the corresponding true object state.
    state = env.reset()
    object_true_state = env.object_manager.get_true_state()
    
    # Call your provided get_mcts_metrics function.
    metrics = get_mcts_metrics(state, object_true_state,
                               max_actions=max_actions,
                               LI=LI,
                               EF=EF,
                               DF=DF,
                               rollout_method=rollout_method,
                               hertg_method=hertg_method,
                               rollout_pre_collision_stop=rollout_pre_collision_stop)
    return metrics

# -----------------------------------------------------------------------------
# Experiment runner: submits a series of tasks in parallel for each rollout method.
# -----------------------------------------------------------------------------
def run_experiments_for_rollout_methods(rollout_methods, num_trials=10, **kwargs):
    """
    For each rollout method in rollout_methods, run num_trials independent experiments.
    Additional parameters for get_mcts_metrics can be passed via kwargs.
    
    Returns:
        A list of dictionaries containing the metrics for each run.
    """
    all_results = []
    with concurrent.futures.ProcessPoolExecutor() as executor:
        futures = []
        # For each rollout method, schedule num_trials experiments.
        for method in rollout_methods:
            for _ in range(num_trials):
                # Submit the task to the process pool.
                futures.append(executor.submit(worker_wrapper, method, **kwargs))
        
        # Collect the results as they complete.
        for future in concurrent.futures.as_completed(futures):
            try:
                result = future.result()
                all_results.append(result)
            except Exception as e:
                print("An error occurred during execution:", e)
    return all_results

# -----------------------------------------------------------------------------
# Results compilation: create a DataFrame from the list of result dictionaries.
# -----------------------------------------------------------------------------
def compile_results_to_dataframe(results):
    """
    Converts the list of metric dictionaries into a Pandas DataFrame.
    """
    return pd.DataFrame(results)

# -----------------------------------------------------------------------------
# Averaging results: group by fixed parameters and average numeric metrics.
# -----------------------------------------------------------------------------
def average_results(df, groupby_cols=['rollout_method', 'LI', 'EF', 'DF', 'hertg_method']):
    """
    Given a DataFrame of results, group by the parameter columns and compute the
    average of the numeric metric columns.
    """
    # Identify numeric columns to average.
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    avg_df = df.groupby(groupby_cols)[numeric_cols].mean().reset_index()
    return avg_df

# -----------------------------------------------------------------------------
# Main function: defines the experiment and saves/prints the results.
# -----------------------------------------------------------------------------
# Define the rollout methods you want to test.
rollout_methods = ['random', 'same', 'random_same', 'zero', 'accelerate']

# Set the number of trials per rollout method.
num_trials = 10  # Adjust as needed for better averaging.

# Run the experiments. You can pass additional parameters (like max_actions, etc.)
results = run_experiments_for_rollout_methods(rollout_methods, num_trials=num_trials)

# Compile the individual run results into a DataFrame.
df = compile_results_to_dataframe(results)
print("Individual run metrics:")
display(df)

# Compute average metrics per parameter configuration.
avg_df = average_results(df)
print("\nAveraged metrics:")
display(avg_df)

# Optionally, save the results to CSV files.
df.to_csv("mcts_metrics_individual.csv", index=False)
avg_df.to_csv("mcts_metrics_averaged.csv", index=False)

# -----------------------------------------------------------------------------
# Run the main function.
# -----------------------------------------------------------------------------
# if __name__ == '__main__':
#     main()


Traceback (most recent call last):
  File "c:\Users\austi\AppData\Local\Programs\Python\Python311\Lib\multiprocessing\queues.py", line 246, in _feed
    send_bytes(obj)
  File "c:\Users\austi\AppData\Local\Programs\Python\Python311\Lib\multiprocessing\connection.py", line 184, in send_bytes
    self._check_closed()
  File "c:\Users\austi\AppData\Local\Programs\Python\Python311\Lib\multiprocessing\connection.py", line 137, in _check_closed
    raise OSError("handle is closed")
OSError: handle is closed
Traceback (most recent call last):
  File "c:\Users\austi\AppData\Local\Programs\Python\Python311\Lib\multiprocessing\queues.py", line 246, in _feed
    send_bytes(obj)
  File "c:\Users\austi\AppData\Local\Programs\Python\Python311\Lib\multiprocessing\connection.py", line 184, in send_bytes
    self._check_closed()
  File "c:\Users\austi\AppData\Local\Programs\Python\Python311\Lib\multiprocessing\connection.py", line 137, in _check_closed
    raise OSError("handle is closed")
OSError: han

OSError: handle is closed